# Olist Reviews Agent

Feature para ajudar sellers do marketplace a monitorar reviews e priorizar ações.

- dashboards de review_score e percentual de avaliações negativas
- alertas quando score < 4 cresce
- análise de texto para identificar categorias de reclamação
  - atraso, defeito, faltando, produto errado, qualidade ruim
- recomendações rápidas para seller agir em logística, embalagem, descrição e prazo de entrega


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes de reviews

Será usada a tabela de reviews e os itens/pedidos relacionados para cruzar sellers e categorias.

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('reviews', reviews.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('sellers', sellers.shape)

## Análise de distribuição de avaliações

Analisar a distribuição completa de notas de review e destacar que scores abaixo de 4 são pontos de preocupação, enquanto 4 e 5 representam clientes mais satisfeitos.


In [ ]:
negative_reviews = reviews[reviews['review_score'] < 4].copy()
negative_reviews['review_text'] = negative_reviews['review_comment_message'].fillna('').str.lower()

review_counts = reviews['review_score'].value_counts().sort_index()
review_pct = review_counts / review_counts.sum() * 100

plt.figure(figsize=(8, 5))
sns.barplot(x=review_counts.index, y=review_pct.values, palette='Blues')
plt.title('Distribuição de notas de avaliação (%)')
plt.xlabel('Review score')
plt.ylabel('Porcentagem (%)')
plt.ylim(0, min(100, max(review_pct.values) * 1.1))
for i, pct in enumerate(review_pct.values):
    plt.text(i, pct + 1, f'{pct:.1f}%', ha='center')
plt.tight_layout()
plt.show()

print('Observação: notas abaixo de 4 são pontos de atenção; notas 4 e 5 indicam maior satisfação do cliente.')


### Classificação de tipos de reclamação

Agrupar palavras-chave em categorias como atraso, defeito e faltando.

In [ ]:
keywords = {
    'atraso': ['atraso', 'atrasou', 'entrega atrasada', 'demora', 'entregue tarde', 'late', 'não chegou'],
    'defeito': ['defeito', 'quebrado', 'danificado', 'estragado', 'avaria', 'não funciona', 'funcionando'],
    'faltando': ['faltando', 'falta', 'não veio', 'ausente', 'faltou', 'incompleto'],
    'produto errado': ['produto errado', 'errado', 'não era', 'não corresponde', 'item errado'],
    'qualidade ruim': ['qualidade ruim', 'ruim', 'pior', 'péssimo', 'decepcionante'],
}

# 1. Armazenar o objeto ax retornado pelo seaborn
ax = sns.barplot(data=issue_counts, x='count', y='issue_type', palette='rocket')

plt.title('Tipos de reclamação em avaliações negativas')
plt.xlabel('Quantidade')
plt.ylabel('Tipo de reclamação')

# 2. Calcular o total e adicionar os rótulos de porcentagem
total = issue_counts['count'].sum()

for p in ax.patches:
  width = p.get_width()
  if width > 0:  # Garante que não processe barras vazias
    percentage = f'{(width / total) * 100:.1f}%'  # Formata com 1 casa decimal

    # Posiciona o texto logo após o término da barra horizontal
    ax.annotate(
        percentage,
        (width, p.get_y() + p.get_height() / 2),
        xytext=(5, 0),  # Deslocamento horizontal de 5 pixels
        textcoords='offset points',
        ha='left',
        va='center',
        fontsize=10,
        color='black',
        weight='bold',
    )

plt.tight_layout()
plt.show()

## Cruzamento com sellers e categorias

Associar as avaliações negativas a sellers e produtos para identificar áreas problemáticas.

In [ ]:
merged_reviews = pd.merge(negative_reviews, order_items, on='order_id', how='left')
merged_reviews = pd.merge(merged_reviews, products[['product_id', 'product_category_name']], on='product_id', how='left')
merged_reviews = pd.merge(merged_reviews, sellers[['seller_id', 'seller_label']], on='seller_id', how='left')

seller_issues = (
    merged_reviews.groupby(['seller_label', 'issue_type'])
    .size()
    .unstack(fill_value=0)
)
seller_issues['total_negative'] = seller_issues.sum(axis=1)

top_sellers_negative = seller_issues.sort_values('total_negative', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_sellers_negative['total_negative'], y=top_sellers_negative.index, palette='magma')
plt.title('Top 10 sellers com mais avaliações negativas')
plt.xlabel('Total de avaliações negativas')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Categorias com mais reclamações negativas

In [ ]:
category_issues = (
    merged_reviews.groupby('product_category_name')['issue_type']
    .count()
    .reset_index(name='negative_reviews')
    .sort_values('negative_reviews', ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=category_issues, x='negative_reviews', y='product_category_name', palette='viridis')
plt.title('Top 10 categorias com mais avaliações negativas')
plt.xlabel('Quantidade de avaliações negativas')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

## Exemplo de reviews negativas classificadas

In [ ]:
negative_reviews[['review_score', 'issue_type', 'review_text']].head(100)